In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1:
# Load the CSV file
df_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(df_path)


In [ ]:
# Task 2:
#Inspect the first few rows
print(f"Shape: {df_food.shape}")
df_food.head()

In [ ]:
# Task 3:
# Display dataset information
df_food.info()

In [ ]:
# Task 4:
# Descriptive statistics for numerical columns
df_food.describe()

In [ ]:
# Task 5:
# Delivery_Time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1:
# Drop the 'Order_ID' column from the data
df_clean = df_food.drop('Order_ID', axis=1).copy()
print(f"Shape after cleaning: {df_clean.shape}")

In [ ]:
# Task 2:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

# For Handle Missing Values
categorical_cols = df_food.select_dtypes(include=["object"]).columns

# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in [categorical_cols]:
    df_clean[col] = df_clean[col].fillna('unknown')

# Fill numrical columns with the mean
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mode()[0])
df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].mode()[0])


check_missing_values(df_clean)

In [ ]:
# Task 3:
# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4:
from sklearn.preprocessing import LabelEncoder , StandardScaler

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean


In [ ]:
# Task 5:
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6:
import seaborn as sns
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

In [ ]:
# Task 1:
# Split the dataset into features (X) and target (y)
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestClassifier

# Use the correct split: KFold
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for linear regression results for each fold
lr_losses = []
lr_mse = []
lr_rmse = []
lr_r2 = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)
  model.fit(X_train, y_train)
  print("Model trained!")

  # Validate
  # Predictions and metrics
  y_pred = model.predict(X_test)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_mse.append(mse)
  lr_rmse.append(rmse)
  lr_r2.append(r2)


In [ ]:
# Task 1:
# Plot for the feature importance from your trained model

# Define features and target
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2:
# Plot predicted delivery time histogram
plt.hist(y_pred, bins=30, edgecolor='black')

plt.title("Predicted Target Distribution")
plt.xlabel("Predicted Value")
plt.ylabel("Frequency")
plt.grid(False)
plt.show()

In [ ]:
# Task Bonus: Write your code here: